# Additional Data

This notebook prepares bureau and installment history as **21 additional features**, with one row per applicant, in `history_feature_artifacts/additional_data_features.csv`.

These features provide additional information for testing generalization under a meaningful nonrandom split. They do not determine the split or guarantee better generalization.

This notebook handles history cleaning, aggregation, export, and a merge check. Application preprocessing, split selection, model training, and evaluation happen after preparing this data. Fit imputation and scaling on training rows only, after the chosen split.

Run top to bottom with `application_train.csv`, `bureau.csv`, and `installments_payments.csv` beside the notebook, or in `/content` on Colab.

## 1. Import packages

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

## 2. Locate the data

Application IDs are used only to check the later merge and feature coverage. No target labels are needed for preparing the history features.

In [2]:
data_path = Path('application_train.csv')
if not data_path.exists():
    data_path = Path('/content/application_train.csv')

application_ids = pd.read_csv(data_path, usecols=['SK_ID_CURR'])
assert application_ids['SK_ID_CURR'].is_unique
print('Application applicants:', len(application_ids))

Application applicants: 307511


## 3. Prepare bureau history

`bureau.csv` describes an applicant's loans with other lenders. We summarize loan counts, active-loan share, recency, overdue days, and credit/debt amounts. Monetary features use only the dataset's `currency 1` records so we do not combine different currencies.

Keep loans opened and updated at or before the application date. Each applicant gets one row, so merging cannot duplicate application rows. Unavailable measurements stay missing; the separate presence flag records whether eligible history exists.

In [3]:
bureau = pd.read_csv(data_path.with_name('bureau.csv'), usecols=[
    'SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'DAYS_CREDIT',
    'DAYS_CREDIT_UPDATE', 'CREDIT_DAY_OVERDUE', 'CREDIT_CURRENCY',
    'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT'
])
bureau = bureau.loc[
    (bureau['DAYS_CREDIT'] <= 0) & (bureau['DAYS_CREDIT_UPDATE'] <= 0)
].copy()
bureau['IS_ACTIVE'] = bureau['CREDIT_ACTIVE'].eq('Active').astype(float)
bureau.loc[bureau['CREDIT_ACTIVE'].isna(), 'IS_ACTIVE'] = np.nan
bureau['CREDIT_CURRENCY1'] = bureau['AMT_CREDIT_SUM'].where(bureau['CREDIT_CURRENCY'].eq('currency 1'))
bureau['DEBT_CURRENCY1'] = bureau['AMT_CREDIT_SUM_DEBT'].where(bureau['CREDIT_CURRENCY'].eq('currency 1'))

bureau_features = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'nunique'),
    BUREAU_ACTIVE_RATE=('IS_ACTIVE', 'mean'),
    BUREAU_LATEST_CREDIT_DAY=('DAYS_CREDIT', 'max'),
    BUREAU_MEAN_CREDIT_DAY=('DAYS_CREDIT', 'mean'),
    BUREAU_MEAN_OVERDUE_DAYS=('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_MAX_OVERDUE_DAYS=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_MEAN_CREDIT_CURRENCY1=('CREDIT_CURRENCY1', 'mean'),
    BUREAU_MAX_CREDIT_CURRENCY1=('CREDIT_CURRENCY1', 'max'),
    BUREAU_MEAN_DEBT_CURRENCY1=('DEBT_CURRENCY1', 'mean'),
    BUREAU_MAX_DEBT_CURRENCY1=('DEBT_CURRENCY1', 'max')
).reset_index()
bureau_features['BUREAU_HAS_HISTORY'] = 1
assert bureau_features['SK_ID_CURR'].is_unique
del bureau
print('Applicants with bureau history:', len(bureau_features))
bureau_features.head()

Applicants with bureau history: 305810


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_RATE,BUREAU_LATEST_CREDIT_DAY,BUREAU_MEAN_CREDIT_DAY,BUREAU_MEAN_OVERDUE_DAYS,BUREAU_MAX_OVERDUE_DAYS,BUREAU_MEAN_CREDIT_CURRENCY1,BUREAU_MAX_CREDIT_CURRENCY1,BUREAU_MEAN_DEBT_CURRENCY1,BUREAU_MAX_DEBT_CURRENCY1,BUREAU_HAS_HISTORY
0,100001,7,0.428571,-49,-735.000000,0.0,0,207623.571429,378000.0,85240.928571,373239.0,1
1,100002,8,0.250000,-103,-874.000000,0.0,0,108131.945625,450000.0,49156.200000,245781.0,1
2,100003,4,0.250000,-606,-1400.750000,0.0,0,254350.125000,810000.0,0.000000,0.0,1
3,100004,2,0.000000,-408,-867.000000,0.0,0,94518.900000,94537.8,0.000000,0.0,1
4,100005,3,0.666667,-62,-190.666667,0.0,0,219042.000000,568800.0,189469.500000,543087.0,1


## 4. Prepare installment-payment history

`installments_payments.csv` describes payments on previous Home Credit loans. Some installments have several payment parts, so we add those parts together **before** calculating payment ratios. We group by applicant, previous loan, installment version/number, and due date. Schedule versions remain separate because this table does not identify one final version.

Only payments and due dates at or before the current application are used. Missing or negative amounts are excluded. Conflicting scheduled amounts within a group make its payment ratio missing. A recorded shortfall means the recorded payments are below the scheduled amount; it does not prove that the borrower ultimately failed to pay. Installments with no payment record are not represented here.

This is the largest CSV. Reading only the required columns and releasing the raw table after grouping reduces memory use.

In [4]:
installments = pd.read_csv(data_path.with_name('installments_payments.csv'), usecols=[
    'SK_ID_CURR', 'SK_ID_PREV', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER',
    'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT'
], dtype={
    'SK_ID_CURR': 'int32', 'SK_ID_PREV': 'int32',
    'NUM_INSTALMENT_VERSION': 'float32', 'NUM_INSTALMENT_NUMBER': 'int32',
    'DAYS_INSTALMENT': 'float32', 'DAYS_ENTRY_PAYMENT': 'float32'
})
installments = installments.loc[
    (installments['DAYS_INSTALMENT'] <= 0)
    & (installments['DAYS_ENTRY_PAYMENT'] <= 0)
    & (installments['AMT_INSTALMENT'] >= 0)
    & (installments['AMT_PAYMENT'] >= 0)
].copy()
installments['LATE_PAYMENT'] = (
    installments['DAYS_ENTRY_PAYMENT'] > installments['DAYS_INSTALMENT']
).astype('int8')
installments['PAYMENT_DELAY'] = (
    installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
).clip(lower=0)

payments = installments.groupby([
    'SK_ID_CURR', 'SK_ID_PREV', 'NUM_INSTALMENT_VERSION',
    'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT'
], sort=False).agg(
    PAID=('AMT_PAYMENT', 'sum'),
    SCHEDULED_MIN=('AMT_INSTALMENT', 'min'),
    SCHEDULED_MAX=('AMT_INSTALMENT', 'max'),
    LATE_PAYMENT=('LATE_PAYMENT', 'max'),
    PAYMENT_DELAY=('PAYMENT_DELAY', 'max')
).reset_index()
del installments

payments['SCHEDULED'] = payments['SCHEDULED_MAX'].where(
    payments['SCHEDULED_MIN'].eq(payments['SCHEDULED_MAX'])
)
payments['PAYMENT_RATIO'] = payments['PAID'] / payments['SCHEDULED'].replace(0, np.nan)
payments['RECORDED_SHORTFALL'] = (
    payments['PAID'] < payments['SCHEDULED'] - 0.01
).astype(float)
payments.loc[payments['SCHEDULED'].isna(), 'RECORDED_SHORTFALL'] = np.nan

installment_features = payments.groupby('SK_ID_CURR').agg(
    INST_SCHEDULE_COUNT=('PAID', 'size'),
    INST_LOAN_COUNT=('SK_ID_PREV', 'nunique'),
    INST_LATE_PAYMENT_RATE=('LATE_PAYMENT', 'mean'),
    INST_MEAN_PAYMENT_DELAY=('PAYMENT_DELAY', 'mean'),
    INST_MAX_PAYMENT_DELAY=('PAYMENT_DELAY', 'max'),
    INST_MEAN_PAYMENT_RATIO=('PAYMENT_RATIO', 'mean'),
    INST_RECORDED_SHORTFALL_RATE=('RECORDED_SHORTFALL', 'mean'),
    INST_MEAN_PAYMENT_AMOUNT=('PAID', 'mean'),
    INST_LATEST_DUE_DAY=('DAYS_INSTALMENT', 'max')
).reset_index()
installment_features['INST_HAS_HISTORY'] = 1
assert installment_features['SK_ID_CURR'].is_unique
del payments
print('Applicants with installment history:', len(installment_features))
installment_features.head()

Applicants with installment history: 339578


,SK_ID_CURR,INST_SCHEDULE_COUNT,INST_LOAN_COUNT,INST_LATE_PAYMENT_RATE,INST_MEAN_PAYMENT_DELAY,INST_MAX_PAYMENT_DELAY,INST_MEAN_PAYMENT_RATIO,INST_RECORDED_SHORTFALL_RATE,INST_MEAN_PAYMENT_AMOUNT,INST_LATEST_DUE_DAY,INST_HAS_HISTORY
0,100001,7,2,0.142857,1.571429,11.0,1.0,0.0,5885.132143,-1619.0,1
1,100002,19,1,0.000000,0.000000,0.0,1.0,0.0,11559.247105,-25.0,1
2,100003,25,3,0.000000,0.000000,0.0,1.0,0.0,64754.586000,-536.0,1
3,100004,3,1,0.000000,0.000000,0.0,1.0,0.0,7096.155000,-724.0,1
4,100005,9,1,0.111111,0.111111,1.0,1.0,0.0,6240.205000,-466.0,1


## 5. Save the additional data

Join the two summaries into a reusable applicant-level feature table. A presence flag of 0 means no eligible history in that source; other missing values stay missing. Missing history does not prove a clean repayment record.

The CSV includes all applicants represented in either history source, allowing it to be reused across different application subsets. Re-running this cell replaces the generated CSV. No target, median imputation, or standard scaling is included.

In [5]:
history_features = bureau_features.merge(
    installment_features, on='SK_ID_CURR', how='outer', validate='one_to_one'
)
history_flags = ['BUREAU_HAS_HISTORY', 'INST_HAS_HISTORY']
history_features[history_flags] = history_features[history_flags].fillna(0).astype(int)
history_numeric_cols = [c for c in history_features if c not in ['SK_ID_CURR', *history_flags]]
assert history_features['SK_ID_CURR'].notna().all()
assert history_features['SK_ID_CURR'].is_unique
assert 'TARGET' not in history_features
assert not np.isinf(history_features.drop(columns='SK_ID_CURR').to_numpy()).any()

output_dir = Path('history_feature_artifacts')
output_dir.mkdir(exist_ok=True)
feature_path = output_dir / 'additional_data_features.csv'
history_features.to_csv(feature_path, index=False)
print('Saved:', feature_path)
print('History applicants:', len(history_features))
print('Numeric measurements:', len(history_numeric_cols))
print('Presence flags:', len(history_flags))

Saved: history_feature_artifacts/additional_data_features.csv
History applicants: 353692
Numeric measurements: 19
Presence flags: 2


## 6. Check the merge

A left join keeps every application ID, in its original row order and index. Applicants absent from both history sources receive presence flags of 0. The measurements remain missing until training-only preprocessing handles them.

In [6]:
merge_check = application_ids.join(
    history_features.set_index('SK_ID_CURR'), on='SK_ID_CURR', validate='one_to_one'
)
merge_check[history_flags] = merge_check[history_flags].fillna(0).astype(int)
assert len(merge_check) == len(application_ids)
assert merge_check.index.equals(application_ids.index)
assert merge_check['SK_ID_CURR'].equals(application_ids['SK_ID_CURR'])
print('Merge check passed.')
print('History coverage among application applicants:')
print(merge_check[history_flags].mean().rename('fraction_with_history'))
merge_check.head()

Merge check passed.
History coverage among application applicants:
BUREAU_HAS_HISTORY    0.856847
INST_HAS_HISTORY      0.948373
Name: fraction_with_history, dtype: float64


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_RATE,BUREAU_LATEST_CREDIT_DAY,BUREAU_MEAN_CREDIT_DAY,BUREAU_MEAN_OVERDUE_DAYS,BUREAU_MAX_OVERDUE_DAYS,BUREAU_MEAN_CREDIT_CURRENCY1,BUREAU_MAX_CREDIT_CURRENCY1,BUREAU_MEAN_DEBT_CURRENCY1,BUREAU_MAX_DEBT_CURRENCY1,BUREAU_HAS_HISTORY,INST_SCHEDULE_COUNT,INST_LOAN_COUNT,INST_LATE_PAYMENT_RATE,INST_MEAN_PAYMENT_DELAY,INST_MAX_PAYMENT_DELAY,INST_MEAN_PAYMENT_RATIO,INST_RECORDED_SHORTFALL_RATE,INST_MEAN_PAYMENT_AMOUNT,INST_LATEST_DUE_DAY,INST_HAS_HISTORY
0,100002,8.0,0.25,-103.0,-874.00,0.0,0.0,108131.945625,450000.0,49156.2,245781.0,1,19.0,1.0,0.000000,0.0,0.0,1.0,0.0,11559.247105,-25.0,1
1,100003,4.0,0.25,-606.0,-1400.75,0.0,0.0,254350.125000,810000.0,0.0,0.0,1,25.0,3.0,0.000000,0.0,0.0,1.0,0.0,64754.586000,-536.0,1
2,100004,2.0,0.00,-408.0,-867.00,0.0,0.0,94518.900000,94537.8,0.0,0.0,1,3.0,1.0,0.000000,0.0,0.0,1.0,0.0,7096.155000,-724.0,1
3,100006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,16.0,3.0,0.000000,0.0,0.0,1.0,0.0,62947.088438,-11.0,1
4,100007,1.0,0.00,-1149.0,-1149.00,0.0,0.0,146250.000000,146250.0,0.0,0.0,1,63.0,5.0,0.253968,1.0,12.0,1.0,0.0,12795.682143,-14.0,1


## 7. Use the additional data

**Immediately after `df = pd.read_csv(data_path)`**, before creating `applicant_ids`, `y`, or `X`, add:

```python
history_features = pd.read_csv('history_feature_artifacts/additional_data_features.csv')
history_flags = ['BUREAU_HAS_HISTORY', 'INST_HAS_HISTORY']
history_numeric_cols = [
    c for c in history_features if c not in ['SK_ID_CURR', *history_flags]
]

df = df.join(
    history_features.set_index('SK_ID_CURR'),
    on='SK_ID_CURR', validate='one_to_one'
)
df[history_flags] = df[history_flags].fillna(0).astype(int)
```

On Colab, upload the generated CSV and adjust that path to its location.

**After the existing `scale_cols` and `binary_cols` lists, before adding missing-value flags**, add:

```python
scale_cols = scale_cols + history_numeric_cols
binary_cols = binary_cols + history_flags
```

The added columns then flow through the existing missing-value flags, median imputation, standard scaling, and neural network. The categorical application columns stay in their existing pipeline. Keep `TARGET` and all ID columns out of model inputs. Running a join alone is insufficient: the explicit preprocessing lists must also include the new columns.

## 8. How this supports the nonrandom-split comparison

Choose the split rule based on a concrete generalization question, before inspecting its model scores. For example, holding out larger-loan applicants tests transfer to larger loans. It represents deployment only if that is the applicant population the deployed system is expected to encounter; otherwise describe it as a distribution-shift stress test.

For the application-only versus additional-data comparison:

1. Keep the exact same applicant IDs in training, validation, and test for both models. Never split the raw history rows separately.
2. Compare the split sizes, feature distributions, target rates, and history coverage to understand which population changed. Fit imputers/scalers only on training rows.
3. Use the same network and training settings, then compare validation ROC AUC, F1, precision/recall, and fairness. Tune thresholds on validation, not test.
4. Document whether history helps under that fixed split. Keep final test evaluation until model decisions are complete.

The random, stratified, and meaningful nonrandom split comparison remains a separate evaluation question from whether these additional features help. Earlier stratified-split scores do not establish performance under a nonrandom split. This preparation notebook reports no model scores.